# Ecom LLM `First notebook` (Modular + Magic Commands)

This notebook is now a thin orchestrator.
- Core logic is implemented in `ecom_llm.py`
- The module is loaded with notebook magic (`%run`)
- Run cells in order

<div class="alert alert-block alert-info">
<b>Note:</b> Since this was designed as a complete project, the code is modular and the different notebooks tackle each stage. 
</div>

<div class="alert alert-block alert-warning">
<b>Part 1:</b>  Criterion definitions is self contained in llm_config.py prompt (Before updates in "LEGACY" and after in the used prompt). Moreover,Pass / fail definition is self contained in llm_scoring.py final_pass_fail() function.
</div>

<div class="alert alert-block alert-warning">
<b>Part 2:</b>  This first notebook tackles the second part without optimizing. The last cell details the next steps.
</div>

<div class="alert alert-block alert-warning">
<b>Part 3:</b>  initial part 3 is Covered here as well.
</div>

<div class="alert alert-block alert-warning">
<b>Part 5:</b>  Self contained in llm_judge.py
</div>

<div class="alert alert-block alert-warning">
<b>Part 6:</b> We'll run the LLM as a judge in all 4 notebooks in this project. In this notebook, pt2 notebook (new_prompt), and pt4 (judging_llm) we'll compare to our manual non-bot human judging. in pt4 (Judging_llm) we touch the last aspect, Criterion-by-criterion judging and compare it to the full criteria prompt.
</div>

In [1]:
# Optional dependency install
%pip install -q openai pandas tqdm pydantic openpyxl

Note: you may need to restart the kernel to use updated packages.


In [4]:
# Load all modular functions/classes into notebook scope using magic command
%run ./ecom_llm.py

In [ ]:
import os
import pandas as pd 
import numpy as np
import re
from IPython.display import HTML, display
from sklearn.metrics import confusion_matrix, classification_report
from llm_scoring import final_pass_fail

os.environ["NEBIUS_API_KEY"] = "Your_key"


## **Full pipeline run**

In [ ]:
data_path = "data/ecommerce_dataset.csv"

results_df, full_df, summary_overview, summary_distribution = run_assignment(
    df=None,
    excel_path="ecommerce_sheet.xlsx",
    data_path=data_path,
)

print("Saved and updated: ecommerce_sheet.xlsx")
print(f"Loaded dataset from: {data_path}")
print("results_df (trimmed output columns):")
display(results_df.head())
print("full_df (includes explanation columns):")
display(full_df.head())

Task 2: Generating:   0%|          | 0/50 [00:00<?, ?it/s]

Task 5: Judging:   0%|          | 0/50 [00:00<?, ?it/s]

Saved and updated: ecommerce_sheet.xlsx
Loaded dataset from: data/ecommerce_dataset.csv
results_df (trimmed output columns):


,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,fluency,grammar,tone,length,grounding,latency_rating,cost_rating,final_score
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1‑year limited warranty,Experience the power and elegance of the Apple...,5110,284,114,good,good,good,good,good,bad,good,fail
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S‑Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1‑year limited warranty,Capture every detail with the stunning 200MP c...,1106,284,92,good,good,good,good,good,good,good,pass
2,Google Pixel 8 Pro,"features: Tensor G3 chip, Magic Eraser, 50 MP ...","matte glass back, aluminum frame",1‑year limited warranty,Capture stunning photos with the Google Pixel ...,1243,281,88,ok,good,good,ok,good,good,good,pass
3,Sony WH‑1000XM5 Headphones,"features: active noise cancelling, 30 hr batte...",synthetic leather earcups,1‑year limited warranty,Escape into pure audio bliss with the Sony WH‑...,1336,281,114,ok,good,ok,bad,good,good,good,fail
4,Bose QuietComfort Ultra Earbuds,"features: CustomTune sound calibration, ANC, I...",silicone ear tips,1‑year limited warranty,Experience unparalleled audio bliss with the B...,1034,270,87,ok,good,ok,good,good,good,good,pass


full_df (includes explanation columns):


,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,fluency,grammar,...,length,grounding,latency_rating,cost_rating,final_score,fluency_explanation,grammar_explanation,tone_explanation,length_explanation,grounding_explanation
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1‑year limited warranty,Experience the power and elegance of the Apple...,5110,284,114,good,good,...,good,good,bad,good,fail,"The description flows naturally, with a good b...",The description is free of spelling and punctu...,"The tone is warm and confident, conveying a se...",The description is within the ideal length ran...,The description accurately reflects the provid...
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S‑Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1‑year limited warranty,Capture every detail with the stunning 200MP c...,1106,284,92,good,good,...,good,good,good,good,pass,"The description flows naturally, with a good b...",The description is free of spelling and punctu...,"The tone is warm and confident, conveying a se...",The description is within the ideal length ran...,The description accurately reflects the provid...
2,Google Pixel 8 Pro,"features: Tensor G3 chip, Magic Eraser, 50 MP ...","matte glass back, aluminum frame",1‑year limited warranty,Capture stunning photos with the Google Pixel ...,1243,281,88,ok,good,...,ok,good,good,good,pass,"The description flows naturally, but there are...",The description has no spelling or punctuation...,The tone of the description is warm and confid...,"The description has 76 words, which is slightl...",The description accurately reflects the provid...
3,Sony WH‑1000XM5 Headphones,"features: active noise cancelling, 30 hr batte...",synthetic leather earcups,1‑year limited warranty,Escape into pure audio bliss with the Sony WH‑...,1336,281,114,ok,good,...,bad,good,good,good,fail,"The description flows naturally, but there are...",The description is free of spelling and punctu...,"The tone is mostly warm and confident, but it'...","The description is 96 words, which is slightly...",The description accurately reflects the provid...
4,Bose QuietComfort Ultra Earbuds,"features: CustomTune sound calibration, ANC, I...",silicone ear tips,1‑year limited warranty,Experience unparalleled audio bliss with the B...,1034,270,87,ok,good,...,good,good,good,good,pass,"The description flows naturally, but there's a...","There are no spelling or punctuation errors, b...","The tone is warm and confident, but it's a bit...","The description is 76 words, which is within t...",All claims are traceable to the provided produ...


***

### Costs calculation

In [32]:
# Add cost column per row to both dataframes
# $0.02 / 1M In
# $0.06 / 1M Out (with caching)
results_df["cost"] = results_df.apply(
    lambda row: 0.02 * row["input_tokens"] / 1e6 + 0.06 * row["output_tokens"] / 1e6,
    axis=1
)
full_df["cost"] = full_df.apply(
    lambda row: 0.02 * row["input_tokens"] / 1e6 + 0.06 * row["output_tokens"] / 1e6,
    axis=1
)

***

#### Saving new df

In [ ]:
results_df.to_excel("data/generated/assignment_01.xlsx", index=False)
full_df.to_excel("data/generated/assignment_01_full.xlsx", index=False)
print("Saved: assignment_01.xlsx and assignment_01_full.xlsx")

Saved: assignment_01.xlsx and assignment_01_full.xlsx


***

### sampling random rows (here only 10)

In [34]:
rows = results_df.sample(n=10, random_state=42).copy()
rows

,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,fluency,grammar,tone,length,grounding,latency_rating,cost_rating,final_score,cost
13,Nintendo Switch OLED,"features: 7″ OLED screen, docked & handheld mo...",plastic chassis,1‑year limited warranty,Experience vibrant gameplay on the go with the...,1127,273,110,good,good,good,ok,good,good,good,pass,0.000012
39,Blink Outdoor 4,"features: 1080p HD, 2‑year battery, weather‑re...",plastic,1‑year limited warranty,Get crystal-clear security with the Blink Outd...,904,272,84,ok,good,ok,ok,good,good,good,pass,0.000010
30,LEGO Star Wars Millennium Falcon 75192,"features: 7 541 pieces, detailed interior, min...",ABS plastic bricks,lifetime replacement of missing bricks,Calling all Star Wars fans! The LEGO Star Wars...,1016,280,102,ok,good,good,ok,good,good,good,pass,0.000012
45,SanDisk Extreme PRO 128 GB SDXC,"features: UHS‑I U3, 200 MB/s read, waterproof;...",plastic,lifetime limited warranty,Capture every detail with the SanDisk Extreme ...,2634,281,103,good,good,good,good,ok,ok,good,pass,0.000012
17,Keurig K‑Supreme Plus Smart,"features: multistream extraction, BrewID, 78 o...",brushed stainless steel accents,1‑year limited warranty,Experience barista-quality coffee at home with...,963,275,90,ok,good,good,good,good,good,good,pass,0.000011
48,BenQ PD2725U 27″ Monitor,"features: 4K UHD IPS, 98% P3, Thunderbolt 3; c...",metal stand,3‑year warranty,Elevate your visual experience with the BenQ P...,1224,282,96,good,good,good,good,good,good,good,pass,0.000011
26,Adidas Ultraboost Light,"features: Light BOOST midsole, Primeknit+ uppe...",textile & synthetic,2‑year manufacturer warranty,Experience the next level of comfort and perfo...,1224,276,115,ok,good,ok,bad,good,good,good,fail,0.000012
25,Nike Air Zoom Pegasus 40,"features: React foam, Zoom Air units, engineer...",textile & synthetic,2‑year manufacturer warranty,Experience the energy-efficient ride of the Ni...,966,270,83,ok,good,good,good,good,good,good,pass,0.000010
32,Kindle Paperwhite 11th Gen,"features: 6.8″ glare‑free display, 10‑week bat...",recycled plastic,1‑year limited warranty,Immerse yourself in a world of reading with th...,1472,281,141,good,good,good,ok,good,good,good,pass,0.000014
19,Dyson V15 Detect Vacuum,"features: laser dust detection, HEPA filtratio...",polycarbonate & aluminum,2‑year parts & labor,The Dyson V15 Detect Vacuum is a powerful clea...,1100,275,114,ok,good,ok,bad,good,good,good,fail,0.000012


***

### Length Analysis

*As we saw the the length calculation was problematic (spoiler alert), we define functions for Programmatically counting the words*

In [ ]:
def count_words(text: str) -> int:
    # Strip punctuation artifacts, split on whitespace
    return len(re.findall(r'\b\w+\b', text))

def classify_length(word_count: int) -> str:
    if 50 <= word_count <= 90:
        return "good"
    elif (40 <= word_count <= 49) or (91 <= word_count <= 110):
        return "ok"
    else:
        return "bad"

***

### Human-as-a-judge

In [ ]:
rows["generated_description"] = rows["generated_description"].str.replace("\n", "<br>", regex=False)

display(HTML(rows[["product_name","generated_description"]].to_html(escape=False, index=False)))

product_name,generated_description
Nintendo Switch OLED,"Experience vibrant gameplay on the go with the Nintendo Switch OLED. This console boasts a stunning 7” OLED screen for crystal-clear visuals, whether you're docked to your TV or playing in handheld mode. With 64GB of storage, you'll have plenty of space for your favorite games. Crafted with a durable plastic chassis, the Switch OLED is ready for adventures big and small. Backed by a 1-year limited warranty, you can enjoy your gaming experience with peace of mind."
Blink Outdoor 4,"Get crystal-clear security with the Blink Outdoor 4, a limited-edition security camera featuring 1080p HD video. Its weather-resistant design and long-lasting 2-year battery keep it running strong, giving you peace of mind day and night. Crafted from durable plastic, the Blink Outdoor 4 is backed by a 1-year limited warranty."
LEGO Star Wars Millennium Falcon 75192,"Calling all Star Wars fans! The LEGO Star Wars Millennium Falcon 75192 is an epic build with 7,541 pieces, capturing every detail of Han Solo's iconic ship. Explore the intricate interior and bring your favorite characters to life with the included minifigures. Built tough with ABS plastic bricks, this set is sure to last a lifetime. Plus, enjoy peace of mind knowing that LEGO offers a lifetime replacement for any missing bricks."
SanDisk Extreme PRO 128 GB SDXC,"Capture every detail with the SanDisk Extreme PRO 128GB SDXC. This durable plastic card boasts UHS-I U3 and a blazing 200MB/s read speed, ensuring smooth performance for your high-resolution photos and videos. Its waterproof design keeps your memories safe from the elements, while the long-lasting battery ensures you can shoot all day long. Backed by a lifetime limited warranty, the SanDisk Extreme PRO is the reliable companion for your adventures."
Keurig K‑Supreme Plus Smart,"Experience barista-quality coffee at home with the Keurig K-Supreme Plus Smart. This sleek brewer, featuring brushed stainless steel accents, boasts multistream extraction technology and BrewID for consistently delicious cups. Its large 78-ounce reservoir means fewer refills, letting you enjoy more coffee throughout the day. Backed by a 1-year limited warranty, the K-Supreme Plus Smart is your perfect smart brewing companion."
BenQ PD2725U 27″ Monitor,"Elevate your visual experience with the BenQ PD2725U 27″ Monitor. Enjoy stunning clarity and vibrant colors thanks to its 4K UHD IPS display and wide P3 color gamut coverage. Seamlessly connect your devices with the powerful Thunderbolt 3 port. The PD2725U boasts a sleek metal stand and comes with a comprehensive 3-year warranty, providing you with peace of mind and reliable performance."
Adidas Ultraboost Light,"Experience the next level of comfort and performance with the Adidas Ultraboost Light. These shoes feature a light BOOST midsole for responsive cushioning and a Primeknit+ upper for a breathable, sock-like fit. The Continental rubber outsole provides superior grip on a variety of surfaces. Backed by a 2-year manufacturer warranty, the Ultraboost Light is built to deliver exceptional performance for miles to come. Don't just take our word for it, over 95% of customers give these shoes a 4.7-star rating!"
Nike Air Zoom Pegasus 40,"Experience the energy-efficient ride of the Nike Air Zoom Pegasus 40. These shoes are crafted with a breathable engineered mesh upper and feature a responsive combination of React foam and Zoom Air units for a cushioned and springy feel. Made from textile and synthetic materials, the Pegasus 40 is backed by a 2-year manufacturer warranty, giving you peace of mind with every stride."
Kindle Paperwhite 11th Gen,"Immerse yourself in a world of reading with the Kindle Paperwhite 11th Gen. Its spacious 6.8"" glare-free display makes every page a pleasure to read, whether you're lounging in the sun or curled up by the fire. Enjoy weeks of reading on a single charge with the impressive 10-week battery life, and easily transfe

In [ ]:
# Choose random rows from results_df
rows['fluency_human'] = np.array(["good", "good", "ok", "good", "ok", "good", "good", "ok", "good", "ok"])
rows['grammar_human'] = np.array(["good", "good", "good", "good", "good", "good", "good", "good", "good", "good"])
rows['tone_human'] = np.array(["ok", "ok", "good", "ok", "good", "bad", "good", "good", "ok", "ok"])
rows['grounding_human'] = np.array(["good", "good", "good", " good", " good", " good", "bad", " good", " good", " good"])
rows['fluency_match'] = rows['fluency_human'] == rows['fluency']
rows['grammar_match'] = rows['grammar_human'] == rows['grammar']
rows['tone_match'] = rows['tone_human'] == rows['tone']
rows['grounding_match'] = rows['grounding_human'].str.strip() == rows['grounding'].str.strip()
print("Fluency matches:", rows['fluency_match'].sum(), "/", len(rows))
print("Grammar matches:", rows['grammar_match'].sum(), "/", len(rows))
print("Tone matches:", rows['tone_match'].sum(), "/", len(rows))
print("Grounding matches:", rows['grounding_match'].sum(), "/", len(rows))

Fluency matches: 8 / 10
Grammar matches: 10 / 10
Tone matches: 5 / 10
Grounding matches: 8 / 10


*Seems like tone matches the least, in grounding we had a weird generation that the LLM as a judge did not catch. We'll address it.*

In [ ]:
rows["length_human"] = rows["generated_description"].apply(count_words)
rows["true_length_label"] = rows["length_human"].apply(classify_length)
rows["judge_correct"] = rows["length"] == rows["true_length_label"]

# Confusion matrix
print(classification_report(rows["true_length_label"], rows["length"]))

              precision    recall  f1-score   support

         bad       0.00      0.00      0.00         0
        good       1.00      0.44      0.62         9
          ok       0.25      1.00      0.40         1

    accuracy                           0.50        10
   macro avg       0.42      0.48      0.34        10
weighted avg       0.93      0.50      0.59        10



c:\Users\maorb\anaconda3\envs\prez\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\maorb\anaconda3\envs\prez\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\maorb\anaconda3\envs\prez\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


*The length was off which is unexpected since it's raw counts. Looking at the judge length_explanation, we might need to edit the prompt or change the model*

***

In [ ]:
rows.to_excel("data/generated/assignment_01_sample_rows.xlsx", index=False)
print("Saved: data/generated/assignment_01_sample_rows.xlsx")

Saved: assignment_01_sample_rows.xlsx


### Our final human scores

In [ ]:
rows = pd.read_excel("data/generated/assignment_01_sample_rows_new.xlsx")
rows['final_score_human'] = rows.apply(
    lambda row: final_pass_fail(
        fluency=row['fluency_human'],
        grammar=row['grammar_human'],
        tone=row['tone_human'],
        length = row['true_length_label'],
        grounding=row['grounding_human'],
        latency_rating= row['latency_rating'],
        cost_rating=row['cost_rating']
    ),
    axis=1
)

In [8]:
rows[['product_name','final_score_human', 'final_score']]

,product_name,final_score_human,final_score
0,Nintendo Switch OLED,pass,fail
1,Blink Outdoor 4,pass,pass
2,LEGO Star Wars Millennium Falcon 75192,pass,pass
3,SanDisk Extreme PRO 128 GB SDXC,pass,pass
4,Keurig K‑Supreme Plus Smart,pass,pass
5,BenQ PD2725U 27″ Monitor,pass,pass
6,Adidas Ultraboost Light,pass,pass
7,Nike Air Zoom Pegasus 40,pass,pass
8,Kindle Paperwhite 11th Gen,pass,pass
9,Dyson V15 Detect Vacuum,pass,fail


In [ ]:
print("=== Final Summary ===")
display(summary_overview)
display(summary_distribution)

=== Final Summary ===


,metric,value
0,pass_rate_percent,76.0
1,pass_count,38.0
2,total_count,50.0


,good,ok,bad,total
fluency,16,34,0,50
grammar,49,1,0,50
tone,35,15,0,50
length,30,10,10,50
grounding,48,2,0,50
latency_rating,47,2,1,50
cost_rating,50,0,0,50


## **Next steps:**

1. Improving the prompts for all metrics, with emphasis on grounding, fluency and tone, while for length we can use regex instead. Moreover, adding guardrails for llm judge in assessing grounding.

2. Using Bigger models

3. Using LLM as a judge with metric specific prompt.